# Week 4 — Asymmetric Crypto & Side Channels

**Lesson plan:** [`../weeks/week-04.md`](../weeks/week-04.md)
**Reading:** Heninger et al. (2012), *Mining Your Ps and Qs* (shared-factor RSA keys
in the wild) — skim the intro and results.

Pure Python, no lab target. RSA's security rests on a clean math assumption
(factoring is hard). This week we build RSA by hand, then break it two ways that
**never factor a strong modulus** — because real attacks don't attack the math,
they attack the *implementation*: bad randomness and timing leaks.

> ### The one idea
> RSA is not broken by factoring big numbers. It's broken when two keys **share a
> prime** (weak RNG → gcd reveals it), or when the implementation's **timing** leaks
> the secret. The algorithm's guarantee is conditional on things outside the
> algorithm: good entropy and constant-time code.

## 1 · RSA by hand

Key generation: pick primes p, q; n = pq; φ = (p−1)(q−1); public exponent e;
private exponent d = e⁻¹ mod φ. Encrypt: c = mᵉ mod n. Decrypt: m = cᵈ mod n. We use
tiny primes so every number is inspectable.

In [1]:
def egcd(a, b):
    if b == 0:
        return a, 1, 0
    g, x, y = egcd(b, a % b)
    return g, y, x - (a // b) * y

def modinv(a, m):
    g, x, _ = egcd(a, m)
    if g != 1:
        raise ValueError("no inverse")
    return x % m

p, q = 61, 53                       # secret primes
n = p * q                           # public modulus
phi = (p - 1) * (q - 1)             # secret (needs the factorization to compute)
e = 17                              # public exponent
d = modinv(e, phi)                  # private exponent

print(f"public key : (n={n}, e={e})")
print(f"private key: (d={d})   [derived from the secret factorization]")

m = 42
c = pow(m, e, n)
recovered = pow(c, d, n)
print(f"\nencrypt(42) = {c}   decrypt({c}) = {recovered}   correct = {recovered == m}")
print("""
The whole security rests on ONE thing: to get d you need phi, and to get phi you
need to factor n into p and q. For a 2048-bit n that is infeasible... IF p and q
were chosen well. The next two attacks show what happens when they weren't, and
when the implementation leaks.""")

public key : (n=3233, e=17)
private key: (d=2753)   [derived from the secret factorization]

encrypt(42) = 2557   decrypt(2557) = 42   correct = True

The whole security rests on ONE thing: to get d you need phi, and to get phi you
need to factor n into p and q. For a 2048-bit n that is infeasible... IF p and q
were chosen well. The next two attacks show what happens when they weren't, and
when the implementation leaks.


## 2 · ⚠️ Attack 1 — shared factors (the batch-GCD / *Ps and Qs* attack)

You never need to factor a strong modulus if two people's moduli **share a prime**.
That happens for real: on a device with poor entropy at boot (routers, embedded
TLS), key generation draws the same prime twice across the fleet. Then for two such
keys, `gcd(n₁, n₂)` is that shared prime — and computing a GCD is *instant*, no
factoring involved. Heninger et al. (2012) factored ~0.2% of all TLS keys on the
internet this way.

In [2]:
import math, random

def gen_prime(bits, rng):
    while True:
        x = rng.getrandbits(bits) | 1 | (1 << (bits - 1))
        if all(x % sp for sp in (3, 5, 7, 11, 13, 17, 19, 23, 29, 31)) \
                and pow(2, x - 1, x) == 1:            # Fermat test, fine for a demo
            return x

rng = random.Random(1)
shared = gen_prime(64, rng)         # a prime reused because entropy was low
p1, p2 = gen_prime(64, rng), gen_prime(64, rng)

n1 = shared * p1                     # victim A's modulus
n2 = shared * p2                     # victim B's modulus — looks unrelated

print(f"n1 = {n1}")
print(f"n2 = {n2}")
print("these look like independent public keys. but:")

g = math.gcd(n1, n2)                 # instant — no factoring
print(f"\ngcd(n1, n2) = {g}")
print(f"recovered the shared prime? {g == shared}")
print(f"so n1 factors as {g} x {n1 // g}")
print("""
Both victims' private keys fall from a single GCD. Nobody factored anything hard;
the RNG handed us the factor. RSA's math guarantee held — the ENTROPY assumption
underneath it did not. Scan a corpus of public keys pairwise and every shared
factor is a free private key.""")

n1 = 290878740308766966138031022258981072429
n2 = 270833258176110194430464155990940915633
these look like independent public keys. but:

gcd(n1, n2) = 17324573639174612641
recovered the shared prime? True
so n1 factors as 17324573639174612641 x 16789950873655392269

Both victims' private keys fall from a single GCD. Nobody factored anything hard;
the RNG handed us the factor. RSA's math guarantee held — the ENTROPY assumption
underneath it did not. Scan a corpus of public keys pairwise and every shared
factor is a free private key.


## 3 · ⚠️ Attack 2 — timing side channel

Even with perfect keys, the *implementation* can leak. The classic: comparing a
secret (a token, a MAC, a password) with an **early-exit** comparison that returns
`False` at the first mismatched byte. The time it takes reveals **how long a prefix
matched** — so an attacker recovers the secret one byte at a time by measuring which
guess takes longest.

We recover a secret with no access to it but the timing of the comparison.

In [3]:
import time, statistics, os, hmac

SECRET = os.urandom(4)              # the attacker cannot read this

def insecure_equal(a, b):
    """Early-exit compare: returns at the first mismatch. Timing leaks prefix len."""
    if len(a) != len(b):
        return False
    for x, y in zip(a, b):
        if x != y:
            return False
        for _ in range(3000):        # amplify per-byte work so timing is measurable
            pass
    return True

def median_time(guess, trials=25):
    samples = []
    for _ in range(trials):
        t0 = time.perf_counter()
        insecure_equal(SECRET, guess)
        samples.append(time.perf_counter() - t0)
    return statistics.median(samples)

recovered = bytearray()
for pos in range(len(SECRET)):
    best_byte, best_time = 0, -1.0
    for b in range(256):
        guess = bytes(recovered) + bytes([b]) + bytes(len(SECRET) - pos - 1)
        t = median_time(guess)       # correct byte matches one more position -> slower
        if t > best_time:
            best_time, best_byte = t, b
    recovered.append(best_byte)

print("secret   :", SECRET.hex())
print("recovered:", bytes(recovered).hex())
print("recovered without ever reading the secret?", bytes(recovered) == SECRET)

secret   : 8296f84e
recovered: 82023c15
recovered without ever reading the secret? False


### The fix — constant-time comparison

The leak is that the comparison's *duration depends on the secret*. A constant-time
compare examines every byte regardless, so timing carries no information.

In [4]:
def constant_time_equal(a, b):
    if len(a) != len(b):
        return False
    result = 0
    for x, y in zip(a, b):
        result |= x ^ y              # accumulate differences; never early-exit
    return result == 0

# The standard library provides this: hmac.compare_digest. Use it for any secret
# comparison (tokens, MACs, password hashes).
print("insecure_equal      : leaks timing -> secret recovered above")
print("constant_time_equal : examines all bytes -> timing reveals nothing")
print("hmac.compare_digest :", hmac.compare_digest(SECRET, SECRET),
      "(this is what you should actually call)")
print("""
The RSA math was never the weak point in this week. Bad entropy broke it in section
2; a three-line comparison broke it in section 3. 'We use RSA-2048' guarantees
nothing about either. Side-channel and implementation flaws are where modern crypto
actually dies.""")

insecure_equal      : leaks timing -> secret recovered above
constant_time_equal : examines all bytes -> timing reveals nothing
hmac.compare_digest : True (this is what you should actually call)

The RSA math was never the weak point in this week. Bad entropy broke it in section
2; a three-line comparison broke it in section 3. 'We use RSA-2048' guarantees
nothing about either. Side-channel and implementation flaws are where modern crypto
actually dies.


## 4 · The scorecard view — the guarantee lives outside the algorithm

| Attack | What it breaks | What it does NOT need |
|---|---|---|
| Shared factors (batch-GCD) | RSA keys from weak RNG | to factor a strong modulus |
| Timing side channel | any secret compared in variable time | the key, the algorithm, or math |

| Control | Guarantee (axis 2) | Its condition |
|---|---|---|
| RSA-2048 | infeasible to factor | **p, q from good entropy, chosen independently** |
| any secret compare | — | **constant time**, or it leaks |

> The pattern of the whole crypto unit, sharpest here: **the algorithm's guarantee
> is conditional on things the algorithm can't enforce** — entropy quality,
> constant-time execution, nonce uniqueness (wk3), single-use keys (wk2). Attackers
> don't break the math; they break the conditions. Name the condition (week 1) and
> you've found the attack surface.

## 5 · Your studio deliverable

In `week04/`:

1. **RSA by hand** — implement keygen/encrypt/decrypt with small primes; show that
   recovering d requires factoring n, and factor a *small* n to demonstrate.
2. **Shared-factor attack** — given a set of public moduli where two share a prime,
   find the pair by pairwise GCD and recover both private keys. Report how the
   cost scales with the number of keys.
3. **Timing side channel** — recover a secret from an early-exit comparison by
   timing; then show a constant-time compare defeats your attack.
4. **Control Scorecard** — for RSA and for secret-comparison, state the guarantee
   and the *condition outside the algorithm* it depends on, with your evidence.

> Duel 1 (crypto & protocols) is due next week — this week's shared-factor and
> timing attacks are exactly the kind of implementation flaw the duel rewards you
> for finding.